Install and Import Libraries

In [2]:
# Install required packages
!pip install nltk scikit-learn pandas

# Import libraries
import pandas as pd
import numpy as np
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aryan\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aryan\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aryan\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

Load Dataset from URL

In [3]:
url = "https://raw.githubusercontent.com/laxmimerit/twitter-data/master/twitter4000.csv"

df = pd.read_csv(url)

df.head()

,twitts,sentiment
0,is bored and wants to watch a movie any sugge...,0
1,back in miami. waiting to unboard ship,0
2,"@misskpey awwww dnt dis brng bak memoriessss, ...",0
3,ughhh i am so tired blahhhhhhhhh,0
4,@mandagoforth me bad! It's funny though. Zacha...,0


Data Understanding

In [4]:
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nClass Distribution:")
print(df['sentiment'].value_counts())

Dataset Shape: (4000, 2)

Columns:
Index(['twitts', 'sentiment'], dtype='object')

Class Distribution:
sentiment
0    2000
1    2000
Name: count, dtype: int64


NLP Preprocessing

In [5]:
#lowercase
df['twitts'] = df['twitts'].str.lower()

#remove urls
def remove_urls(text):
    return re.sub(r"http\S+|www\S+", "", text)

#Remove Special Characters
def remove_special_chars(text):
    return re.sub(r'[^a-zA-Z\s]', '', text)

#remove puntuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

#Tokenization
def tokenize(text):
    return word_tokenize(text)

#Remove Stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

#Stemming
stemmer = PorterStemmer()

def stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

Reusable NLP Pipeline

In [6]:
def preprocess_text(text):

    text = remove_urls(text)
    text = remove_special_chars(text)
    text = remove_punctuation(text)

    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = stemming(tokens)

    return " ".join(tokens)
     

Preprocessing

In [7]:
df['clean_text'] = df['twitts'].apply(preprocess_text)

df[['twitts','clean_text']].head()

,twitts,clean_text
0,is bored and wants to watch a movie any sugge...,bore want watch movi suggest
1,back in miami. waiting to unboard ship,back miami wait unboard ship
2,"@misskpey awwww dnt dis brng bak memoriessss, ...",misskpey dnt di brng bak memoriessss thnk im s...
3,ughhh i am so tired blahhhhhhhhh,ughhh tire blahhhhhhhhh
4,@mandagoforth me bad! it's funny though. zacha...,mandagoforth bad funni though zachari quinto t...


Feature Engineering

In [8]:
#Bag of Words
bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['clean_text'])

print("BoW Shape:", X_bow.shape)

#TF-IDF
tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(df['clean_text'])

print("TF-IDF Shape:", X_tfidf.shape)
     

BoW Shape: (4000, 5000)
TF-IDF Shape: (4000, 5000)


Train Test Split

In [9]:
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=42
)

Model Building

In [10]:
#Logistic Regression
lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

#Naive Bayes
nb = MultinomialNB()

nb.fit(X_train, y_train)

y_pred_nb = nb.predict(X_test)

#Decision Tree
dt = DecisionTreeClassifier()

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

Model Evaluation

In [11]:
#Logistic Regression
print("Logistic Regression")

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1 Score:", f1_score(y_test, y_pred_lr))

print(classification_report(y_test, y_pred_lr))


#Naive Bayes
print("Naive Bayes")

print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb))
print("Recall:", recall_score(y_test, y_pred_nb))
print("F1 Score:", f1_score(y_test, y_pred_nb))

print(classification_report(y_test, y_pred_nb))


#Decision Tree
print("Decision Tree")

print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt))
print("Recall:", recall_score(y_test, y_pred_dt))
print("F1 Score:", f1_score(y_test, y_pred_dt))

print(classification_report(y_test, y_pred_dt))

Logistic Regression
Accuracy: 0.72125
Precision: 0.6832151300236406
Recall: 0.7645502645502645
F1 Score: 0.7215980024968789
              precision    recall  f1-score   support

           0       0.76      0.68      0.72       422
           1       0.68      0.76      0.72       378

    accuracy                           0.72       800
   macro avg       0.72      0.72      0.72       800
weighted avg       0.73      0.72      0.72       800

Naive Bayes
Accuracy: 0.71
Precision: 0.6901041666666666
Recall: 0.701058201058201
F1 Score: 0.6955380577427821
              precision    recall  f1-score   support

           0       0.73      0.72      0.72       422
           1       0.69      0.70      0.70       378

    accuracy                           0.71       800
   macro avg       0.71      0.71      0.71       800
weighted avg       0.71      0.71      0.71       800

Decision Tree
Accuracy: 0.65125
Precision: 0.6371191135734072
Recall: 0.6084656084656085
F1 Score: 0.622462787

Model Comparison Table

In [12]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    "Precision": [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_nb),
        precision_score(y_test, y_pred_dt)
    ],
    "Recall": [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_nb),
        recall_score(y_test, y_pred_dt)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_nb),
        f1_score(y_test, y_pred_dt)
    ]
})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.72125,0.683215,0.764550,0.721598
1,Naive Bayes,0.71000,0.690104,0.701058,0.695538
2,Decision Tree,0.65125,0.637119,0.608466,0.622463
